# JEPA projector — is it usable, and which checkpoint?

Analyses the Stage-1 projector (I-JEPA ViT-H/16 → MLP 1280→4096→4096 → frozen 4-bit Vicuna-7B-v1.5) across every checkpoint you attach.

**Attach as inputs:** (1) COCO2014 (`nadaibrahim/coco2014`), (2) a dataset holding the checkpoint files you want compared (download specific versions from the `jepa-projector-checkpoints` Versions page in the browser, rename e.g. `ckpt_6600.pt`, upload together). Needs GPU (T4). Internet ON (HF weights).

**Gates per checkpoint**
1. Integrity — NaN/Inf weights, magnitude, duplicate weights, stored `step`
2. Stability — non-finite loss/logits over every POPE-manifest image (the images you will actually run), projector output magnitude vs Vicuna text-embedding norm
3. Grounding — caption loss with the right image vs a shuffled image vs no image (paired t-stat), held-out vs train-split
4. Captions — generated captions scored for object precision/recall against COCO annotations, vs shuffled-image captions
5. Mini-POPE — Yes/No margin AUC (real image vs shuffled-image null), two prompts
6. Overlap — manifest group disjointness; pad-token loss bug size; optional fp16-Vicuna cross-check

Results are written to `/kaggle/working/projector_analysis/` after each checkpoint, so a crash/restart resumes.

In [1]:
!pip install -q "transformers==5.0.0" "accelerate==1.13.0" "bitsandbytes==0.50.2" "pillow==11.3.0" scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 72.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 94.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 68.3 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [2]:
   import torch, transformers, accelerate, bitsandbytes, PIL
   print(transformers.__version__, accelerate.__version__, bitsandbytes.__version__, torch.__version__, PIL.__version__, torch.cuda.get_device_name(0), torch.cuda.device_count())

5.0.0 1.13.0 0.50.2 2.11.0+cu128 12.3.0 Tesla T4 2


In [3]:
import os, json, glob, gzip, base64, time, hashlib, random, re, math
import numpy as np
import torch, torch.nn as nn
from PIL import Image

# ------------------------- CONFIG -------------------------
COCO_ROOT   = "/kaggle/input/datasets/nadaibrahim/coco2014"
IMG_DIR     = f"{COCO_ROOT}/val2014/val2014"
CAP_JSON    = f"{COCO_ROOT}/captions/annotations/captions_val2014.json"
INST_JSON   = f"{COCO_ROOT}/captions/annotations/instances_val2014.json"
CKPT_ROOTS  = ["/kaggle/input"]          # scanned recursively
CKPT_NAME_RE = r"(ckpt|projector|mm_projector)"   # filename filter, avoids loading unrelated .pt caches
ONLY_STEPS  = None                       # e.g. [6600, 6800, 7000]; None = every checkpoint found
LLM_PATH    = "lmsys/vicuna-7b-v1.5"
OUT_DIR     = "/kaggle/working/projector_analysis"

QUICK       = True                      # True = tiny run to test the pipeline (~10 min total)
SCAN_N      = 60  if QUICK else 500      # POPE-manifest images for the NaN/loss scan (max 500)
GROUND_N    = 30  if QUICK else 150      # of those, how many also get shuffled-image loss
TRAIN_N     = 20  if QUICK else 100      # projector_train images (train-split loss)
POPE_IMGS   = 20  if QUICK else 100      # images for mini-POPE (x2 questions each)
GEN_N       = 12  if QUICK else 60       # images for caption generation metrics
GEN_TOPK    = 3                          # generate only for the top-K eligible checkpoints
RUN_FP16_CHECK = False                   # needs 2 GPUs (Kaggle T4 x2): re-runs failing images with fp16 Vicuna
SEED        = 0
# ----------------------------------------------------------
os.makedirs(OUT_DIR, exist_ok=True)
device = "cuda"
assert torch.cuda.is_available(), "Turn on a GPU"
print("GPUs:", torch.cuda.device_count(), torch.cuda.get_device_name(0))

GPUs: 2 Tesla T4


In [4]:
# manifest.json (embedded copy of the one used for training; groups verified disjoint)
B64 = """..."""
manifest = json.loads(gzip.decompress(base64.b64decode(B64)))
print({k: len(v) for k, v in manifest.items()})
ks = list(manifest)
for i in range(len(ks)):
    for j in range(i + 1, len(ks)):
        ov = len(set(manifest[ks[i]]) & set(manifest[ks[j]]))
        if ov: print("OVERLAP", ks[i], ks[j], ov)
print("overlap check done (no OVERLAP lines above = all groups disjoint)")

{'pope': 500, 'tuned_lens_train': 350, 'tuned_lens_val': 75, 'tuned_lens_test': 75, 'projector_train': 5000}
overlap check done (no OVERLAP lines above = all groups disjoint)


In [5]:
# I-JEPA tower, identical to ijepa_encoder.py used in training
from transformers import AutoModel, AutoImageProcessor, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

class IJEPAVisionTower(nn.Module):
    def __init__(self, model_id="facebook/ijepa_vith16_1k", device="cuda"):
        super().__init__(); self.model_id = model_id; self.device_ = device
    def load_model(self):
        self.image_processor = AutoImageProcessor.from_pretrained(self.model_id)
        self.vision_tower = AutoModel.from_pretrained(self.model_id)
        self.vision_tower.requires_grad_(False); self.vision_tower.eval(); self.vision_tower.to(self.device_)
    @torch.no_grad()
    def forward(self, pil_images):
        if not isinstance(pil_images, list): pil_images = [pil_images]
        inputs = self.image_processor(pil_images, return_tensors="pt").to(self.device_)
        return self.vision_tower(**inputs).last_hidden_state
    @property
    def hidden_size(self): return self.vision_tower.config.hidden_size

class MLPProjector(nn.Module):
    def __init__(self, v, l):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(v, l), nn.GELU(), nn.Linear(l, l))
    def forward(self, x): return self.proj(x)

tower = IJEPAVisionTower(device=device); tower.load_model()
tokenizer = AutoTokenizer.from_pretrained(LLM_PATH)
print("pad token:", tokenizer.pad_token, "| padding_side:", tokenizer.padding_side, "| bos:", tokenizer.bos_token, "| eos:", tokenizer.eos_token)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token   # same as training
llm = AutoModelForCausalLM.from_pretrained(
    LLM_PATH, quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
    device_map="auto", low_cpu_mem_usage=True).eval()
for p in llm.parameters(): p.requires_grad_(False)
LLM_DTYPE = next(llm.parameters()).dtype
H_LLM = llm.config.hidden_size
_t = tower([Image.new("RGB", (640, 480))])
print("encoder output:", tuple(_t.shape), _t.dtype, "| expected (1, 784, 1280)")
assert _t.shape[1:] == (784, 1280), "encoder shape differs from training"
emb_w = llm.get_input_embeddings().weight
TEXT_EMB_NORM = emb_w[:32000].float().norm(dim=1).mean().item() if emb_w.dtype != torch.uint8 else float("nan")
print("llm dtype:", LLM_DTYPE, "| mean text-embedding row norm:", round(TEXT_EMB_NORM, 3))

preprocessor_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/497 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.53GB            

model.safetensors: downloading bytes:           |  0.00B            

...

model.safetensors.index.json:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/162 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


encoder output: (1, 784, 1280) torch.float32 | expected (1, 784, 1280)
llm dtype: torch.float16 | mean text-embedding row norm: 1.082


## 1. Checkpoint discovery + integrity (Gate 1)

In [6]:
CKPT_DIR = "/kaggle/input/datasets/ponishbhatia/jepa-candidates"

def torch_load_safe(path):
    try:    return torch.load(path, map_location="cpu", weights_only=True)
    except Exception: return torch.load(path, map_location="cpu", weights_only=False)

def find_ckpts():
    print("contents of", CKPT_DIR)
    found = []
    for dp, _, fs in os.walk(CKPT_DIR):
        for f in sorted(fs):
            p = os.path.join(dp, f); sz = os.path.getsize(p) / 1e6
            print(f"   {p.replace(CKPT_DIR, '.')}  {sz:.0f} MB")
            if f.lower().endswith((".pt", ".bin", ".pth", ".ckpt")) and sz < 600:
                found.append(p)
    return sorted(found)

def state_hash(sd):
    h = hashlib.sha1()
    for k in sorted(sd): h.update(sd[k].detach().cpu().float().numpy().tobytes())
    return h.hexdigest()[:12]

def step_of(label):
    m = re.match(r"s(\d+)", label); return int(m.group(1)) if m else -1

CKPTS, rows, seen = {}, [], {}
for p in find_ckpts():
    stem = os.path.splitext(os.path.basename(p))[0]
    try: obj = torch_load_safe(p)
    except Exception as e: print("skip (cannot load)", stem, "|", e); continue
    sd = obj["projector"] if isinstance(obj, dict) and "projector" in obj else obj
    if not (isinstance(sd, dict) and "proj.0.weight" in sd):
        print("skip (not a projector state_dict)", stem, "| top keys:", list(obj)[:6] if isinstance(obj, dict) else type(obj)); continue
    step = obj.get("step") if isinstance(obj, dict) else None
    if step is None:                                   # bare state_dict: try the number in the filename
        m = re.search(r"(\d{3,5})", stem); step = int(m.group(1)) if m else None
    sd = {k: v.float() for k, v in sd.items() if torch.is_tensor(v)}
    n_nan = sum(int(torch.isnan(v).sum()) for v in sd.values()); n_inf = sum(int(torch.isinf(v).sum()) for v in sd.values())
    fin = [float(v.abs().max()) for v in sd.values() if torch.isfinite(v).all()]
    mx = max(fin) if fin else float("nan")
    h = state_hash(sd)
    dup_of = seen.get(h)
    label = f"s{step}" if step is not None else f"s-1_{stem}"
    if label in CKPTS or label in [r["label"] for r in rows]:      # same stored step, different weights/file
        label = f"{label}_{stem}"
    seen.setdefault(h, label)
    rows.append(dict(label=label, file=os.path.basename(p), stored_step=obj.get("step") if isinstance(obj, dict) else None,
                     nan=n_nan, inf=n_inf, max_abs=round(mx, 3), hash=h, dup_of=dup_of))
    if dup_of is None: CKPTS[label] = sd

print()
for r in rows: print(r)
if ONLY_STEPS: CKPTS = {k: v for k, v in CKPTS.items() if step_of(k) in ONLY_STEPS}
bad = [r["label"] for r in rows if r["nan"] or r["inf"]]
CKPTS = {k: v for k, v in CKPTS.items() if k not in bad}
print("\nanalysing:", list(CKPTS), "| excluded (NaN/Inf weights):", bad,
      "| duplicates ignored:", [(r["label"], r["dup_of"]) for r in rows if r["dup_of"]])
assert CKPTS, "No usable checkpoints: check the directory listing printed above"

contents of /kaggle/input/datasets/ponishbhatia/jepa-candidates
   ./ckpt_6600.pt  264 MB
   ./ckpt_6800.pt  264 MB

{'label': 's6600', 'file': 'ckpt_6600.pt', 'stored_step': 6600, 'nan': 0, 'inf': 0, 'max_abs': 0.329, 'hash': '3e87e7cf9451', 'dup_of': None}
{'label': 's6800', 'file': 'ckpt_6800.pt', 'stored_step': 6800, 'nan': 0, 'inf': 0, 'max_abs': 0.341, 'hash': '4ace87be3198', 'dup_of': None}

analysing: ['s6600', 's6800'] | excluded (NaN/Inf weights): [] | duplicates ignored: []


## 2. Data: held-out POPE-manifest images, train-split images, COCO annotations

In [7]:
rng = random.Random(SEED)
cap_raw = json.load(open(CAP_JSON))
id2f = {im["id"]: im["file_name"] for im in cap_raw["images"]}
first_cap = {}
for a in cap_raw["annotations"]:
    first_cap.setdefault(id2f[a["image_id"]], a["caption"])      # first caption, same as training
del cap_raw

pope_imgs  = manifest["pope"][:SCAN_N]
train_imgs = rng.sample(manifest["projector_train"], TRAIN_N)
need = set(pope_imgs) | set(train_imgs)

inst = json.load(open(INST_JSON))
cat_names = {c["id"]: c["name"] for c in inst["categories"]}
f2id = {im["file_name"]: im["id"] for im in inst["images"]}
id2cats = {}
for a in inst["annotations"]:
    id2cats.setdefault(a["image_id"], set()).add(cat_names[a["category_id"]])
gt_cats = {f: id2cats.get(f2id[f], set()) for f in need}
ALL_CATS = sorted(cat_names.values())
del inst
print(len(pope_imgs), "pope imgs,", len(train_imgs), "train imgs; avg GT categories/img:", np.mean([len(v) for v in gt_cats.values()]).round(2))

# cache frozen-encoder features once (encoder is frozen, identical for every checkpoint)
FEATS = {}
t0 = time.time(); names = sorted(need)
for i in range(0, len(names), 8):
    chunk = names[i:i+8]
    imgs = [Image.open(f"{IMG_DIR}/{n}").convert("RGB") for n in chunk]
    out = tower(imgs).float().cpu()
    for n, o in zip(chunk, out): FEATS[n] = o
print(f"encoded {len(FEATS)} images in {time.time()-t0:.0f}s | cache {len(FEATS)*784*1280*4/1e9:.1f} GB")

60 pope imgs, 20 train imgs; avg GT categories/img: 3.44
encoded 80 images in 28s | cache 0.3 GB


## 3. Core evaluation functions

In [8]:
def make_projector(sd):
    p = MLPProjector(1280, H_LLM).to(device); p.load_state_dict(sd); return p.eval()

@torch.no_grad()
def project(projector, name):
    return projector(FEATS[name].unsqueeze(0).to(device))             # (1,784,4096) fp32

def _embed(ids): return llm.get_input_embeddings()(ids)

@torch.no_grad()
def fwd(img_emb, text, mask_bos=True):
    """Caption LM loss exactly as in training, but batch=1 (no padding) and BOS label masked
    so real / shuffled / no-image losses are comparable. Returns (loss, finite_flag)."""
    ids = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).input_ids.to(device)
    te = _embed(ids)
    if img_emb is None:
        emb, lab = te, ids.clone()
    else:
        emb = torch.cat([img_emb, te], 1)
        lab = torch.cat([torch.full((1, img_emb.shape[1]), -100, device=device, dtype=ids.dtype), ids], 1)
    if mask_bos: lab[:, (0 if img_emb is None else img_emb.shape[1])] = -100
    emb = emb.to(LLM_DTYPE)
    out = llm(inputs_embeds=emb, attention_mask=torch.ones(emb.shape[:2], dtype=torch.long, device=device), labels=lab)
    finite = bool(torch.isfinite(out.logits).all()) and bool(torch.isfinite(out.loss))
    return float(out.loss), finite

YES_IDS = sorted({tokenizer.convert_tokens_to_ids(t) for t in ["\u2581Yes", "\u2581yes", "Yes", "yes"]} - {tokenizer.unk_token_id})
NO_IDS  = sorted({tokenizer.convert_tokens_to_ids(t) for t in ["\u2581No", "\u2581no", "No", "no"]} - {tokenizer.unk_token_id})
print("yes ids", YES_IDS, "no ids", NO_IDS)

SYS = ("A chat between a curious human and an artificial intelligence assistant. "
       "The assistant gives helpful, detailed, and polite answers to the human's questions. ")
PROMPTS = {
    "cap": lambda o: f"Question: Is there a {o} in the image? Answer:",
    "vic": lambda o: f"{SYS}USER: Is there a {o} in the image? Please answer yes or no. ASSISTANT:",
}

@torch.no_grad()
def yes_no_margin(img_emb, prompt):
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    emb = torch.cat([img_emb, _embed(ids)], 1).to(LLM_DTYPE)
    lg = llm(inputs_embeds=emb, attention_mask=torch.ones(emb.shape[:2], dtype=torch.long, device=device)).logits[0, -1].float()
    if not torch.isfinite(lg).all(): return float("nan")
    lp = torch.log_softmax(lg, -1)
    return float(torch.logsumexp(lp[YES_IDS], 0) - torch.logsumexp(lp[NO_IDS], 0))

def make_pope_items(imgs, n_img, seed=1):
    r = random.Random(seed); items = []
    for f in imgs[:n_img]:
        pres = sorted(gt_cats[f]);
        if not pres: continue
        absn = [c for c in ALL_CATS if c not in gt_cats[f]]
        items.append((f, r.choice(pres), 1)); items.append((f, r.choice(absn), 0))
    return items
POPE_ITEMS = make_pope_items(pope_imgs, POPE_IMGS)
print("mini-POPE questions:", len(POPE_ITEMS), "| positives:", sum(i[2] for i in POPE_ITEMS))

from sklearn.metrics import roc_auc_score
def auc_ci(y, s, n=200, seed=0):
    y, s = np.array(y), np.array(s); ok = np.isfinite(s); y, s = y[ok], s[ok]
    if len(set(y)) < 2: return float("nan"), float("nan"), float("nan")
    a = roc_auc_score(y, s); r = np.random.RandomState(seed); bs = []
    for _ in range(n):
        ix = r.randint(0, len(y), len(y))
        if len(set(y[ix])) == 2: bs.append(roc_auc_score(y[ix], s[ix]))
    return a, np.percentile(bs, 2.5), np.percentile(bs, 97.5)

# ---- caption object scoring (CHAIR-like) ----
SYN = {"person": "person people man men woman women boy girl child children kid kids guy lady player skier surfer rider baby crowd",
       "car": "car cars", "tv": "tv television", "couch": "couch sofa", "dining table": "table tables", "potted plant": "plant plants",
       "cell phone": "phone phones", "motorcycle": "motorcycle motorbike", "bicycle": "bicycle bike bikes", "airplane": "airplane plane jet",
       "sports ball": "ball", "tennis racket": "racket racquet", "baseball bat": "bat", "baseball glove": "glove mitt",
       "donut": "donut doughnut donuts", "refrigerator": "refrigerator fridge", "teddy bear": "teddy",
       "dog": "dog dogs puppy", "cat": "cat cats kitten", "horse": "horse horses", "cow": "cow cows cattle",
       "sheep": "sheep lamb", "elephant": "elephant elephants", "giraffe": "giraffe giraffes", "zebra": "zebra zebras",
       "boat": "boat boats ship", "bus": "bus", "truck": "truck trucks"}
def mentioned_cats(text):
    low = text.lower().replace("teddy bear", "teddy"); words = set(re.findall(r"[a-z]+", low)); out = set()
    for c in ALL_CATS:
        if c in SYN: hit = any(w in words for w in SYN[c].split())
        elif " " in c: hit = c in low
        else: hit = (c in words) or (c + "s" in words)
        if hit: out.add(c)
    return out

@torch.no_grad()
def gen_caption(img_emb, max_new=30):
    ids = tokenizer("", return_tensors="pt").input_ids.to(device)          # just BOS, like training
    emb = torch.cat([img_emb, _embed(ids)], 1).to(LLM_DTYPE)
    out = llm.generate(inputs_embeds=emb, attention_mask=torch.ones(emb.shape[:2], dtype=torch.long, device=device),
                       max_new_tokens=max_new, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    txt = tokenizer.decode(out[0], skip_special_tokens=True).strip()
    return re.split(r"[.\n]", txt)[0].strip()
print("helpers ready")

yes ids [3582, 3869, 4874, 8241] no ids [694, 1217, 1939, 3782]
mini-POPE questions: 40 | positives: 20
helpers ready


## 4. No-image baseline (checkpoint-independent)

In [9]:
G_IMGS = pope_imgs[:GROUND_N]
t0 = time.time()
noimg = {f: fwd(None, first_cap[f])[0] for f in G_IMGS}
print(f"no-image caption loss, held-out: {np.mean(list(noimg.values())):.3f} ({time.time()-t0:.0f}s)")

no-image caption loss, held-out: 11.248 (3s)


checking to see if vicuna is broken

In [11]:
import torch, transformers, accelerate, bitsandbytes, PIL
print(transformers.__version__, accelerate.__version__, bitsandbytes.__version__, torch.__version__, PIL.__version__,
      torch.cuda.is_available(), torch.cuda.device_count())
print("llm dtype:", next(llm.parameters()).dtype)
dm = getattr(llm, "hf_device_map", {}); print("device_map devices:", sorted(set(map(str, dm.values()))))

# plain-text sanity: no image, no projector
ids = tokenizer("The capital of France is Paris.", return_tensors="pt").input_ids.to(device)
with torch.no_grad():
    print("plain sentence loss:", float(llm(input_ids=ids, labels=ids).loss))
    g = llm.generate(tokenizer("The capital of France is", return_tensors="pt").input_ids.to(device), max_new_tokens=8, do_sample=False)
print(tokenizer.decode(g[0]))

5.0.0 1.13.0 0.50.2 2.11.0+cu128 12.3.0 True 2
llm dtype: torch.float16
device_map devices: ['0', '1']
plain sentence loss: 10.135486602783203
<s> The <unk> cap it al <unk> of <unk> Fran ce <unk> is Par is . ▁nobody ▁knows ▁what ▁the ▁capital


In [12]:
s = "A yellow clock that is on the floor."
print(tokenizer.convert_ids_to_tokens(tokenizer(s).input_ids))

['<s>', 'A', '<unk>', 'ye', 'll', 'ow', '<unk>', 'cl', 'ock', '<unk>', 'th', 'at', '<unk>', 'is', '<unk>', 'on', '<unk>', 'the', '<unk>', 'floor', '.']


In [13]:
from transformers import AutoTokenizer, LlamaTokenizer
s = "A yellow clock that is on the floor."
for name, f in [("fast default", lambda: AutoTokenizer.from_pretrained(LLM_PATH)),
                ("use_fast=False", lambda: AutoTokenizer.from_pretrained(LLM_PATH, use_fast=False)),
                ("LlamaTokenizer", lambda: LlamaTokenizer.from_pretrained(LLM_PATH))]:
    try:
        t = f(); print(name, type(t).__name__, t.convert_ids_to_tokens(t(s).input_ids))
    except Exception as e:
        print(name, "FAILED", repr(e)[:150])

fast default TokenizersBackend ['<s>', 'A', '<unk>', 'ye', 'll', 'ow', '<unk>', 'cl', 'ock', '<unk>', 'th', 'at', '<unk>', 'is', '<unk>', 'on', '<unk>', 'the', '<unk>', 'floor', '.']
use_fast=False TokenizersBackend ['<s>', 'A', '<unk>', 'ye', 'll', 'ow', '<unk>', 'cl', 'ock', '<unk>', 'th', 'at', '<unk>', 'is', '<unk>', 'on', '<unk>', 'the', '<unk>', 'floor', '.']
LlamaTokenizer LlamaTokenizer ['<s>', '▁A', '▁yellow', '▁clock', '▁that', '▁is', '▁on', '▁the', '▁floor', '.']


## 5. Main loop — per checkpoint: stability scan, grounding, mini-POPE
Prints an ETA after the first checkpoint. Results saved after each one.

In [10]:
RES_PATH = f"{OUT_DIR}/results.json"
RES = json.load(open(RES_PATH)) if os.path.exists(RES_PATH) else {}
BAD_IMGS = {}

def eval_ckpt(label, sd):
    P = make_projector(sd); R = {}
    # --- stability + held-out real loss over all POPE-manifest images ---
    real, nonfinite, bad, mx, nrm, shuf = {}, 0, [], [], [], {}
    for k, f in enumerate(pope_imgs):
        e = project(P, f)
        pf = bool(torch.isfinite(e).all()) and bool(torch.isfinite(e.to(LLM_DTYPE)).all())
        mx.append(float(e.abs().max())); nrm.append(float(e.norm(dim=-1).mean()))
        l, ok = fwd(e, first_cap[f])
        if not (ok and pf): nonfinite += 1; bad.append(f)
        else: real[f] = l
    R["scan_n"] = len(pope_imgs); R["nonfinite"] = nonfinite; R["nonfinite_imgs"] = bad
    R["worse_than_noimg"] = int(sum(v > np.mean(list(noimg.values())) + 1.0 for v in real.values()))
    R["proj_max_abs_max"] = max(mx); R["proj_max_abs_p99"] = float(np.percentile(mx, 99)); R["proj_tok_norm_mean"] = float(np.mean(nrm))
    R["norm_ratio_vs_text"] = float(np.mean(nrm)) / TEXT_EMB_NORM
    # --- grounding: shuffled-image loss (derangement), paired with real ---
    r = random.Random(SEED + 7); perm = G_IMGS[:]; r.shuffle(perm)
    d_shuf, d_noimg, rl = [], [], []
    for f, g in zip(G_IMGS, perm):
        if f not in real or g == f: continue
        l, ok = fwd(project(P, g), first_cap[f])
        if ok: d_shuf.append(l - real[f]); rl.append(real[f]); d_noimg.append(noimg[f] - real[f])
    R["heldout_loss"] = float(np.mean(list(real.values())))
    R["heldout_loss_ground_subset"] = float(np.mean(rl)) if rl else float("nan")
    R["shuf_minus_real_mean"] = float(np.mean(d_shuf)) if d_shuf else float("nan")
    R["shuf_minus_real_t"] = float(np.mean(d_shuf) / (np.std(d_shuf, ddof=1) / math.sqrt(len(d_shuf)))) if len(d_shuf) > 2 else float("nan")
    R["noimg_minus_real_mean"] = float(np.mean(d_noimg)) if d_noimg else float("nan")
    # --- train-split loss ---
    tl = []
    for f in train_imgs:
        l, ok = fwd(project(P, f), first_cap[f])
        if ok: tl.append(l)
    R["train_loss"] = float(np.mean(tl)) if tl else float("nan"); R["train_nonfinite"] = len(train_imgs) - len(tl)
    # --- mini-POPE ---
    ys = [it[2] for it in POPE_ITEMS]
    perm_i = list(range(len(POPE_ITEMS))); random.Random(SEED + 3).shuffle(perm_i)
    def other_img(f, j):
        for d in range(len(POPE_ITEMS)):
            g = POPE_ITEMS[(perm_i[j] + d) % len(POPE_ITEMS)][0]
            if g != f: return g
    for pname, pf_ in PROMPTS.items():
        sc, sn = [], []
        for j, (f, o, y) in enumerate(POPE_ITEMS):
            sc.append(yes_no_margin(project(P, f), pf_(o)))
            if pname == "cap":      # null: same question, image from a different item
                sn.append(yes_no_margin(project(P, other_img(f, j)), pf_(o)))
        a, lo, hi = auc_ci(ys, sc); R[f"pope_{pname}_auc"] = float(a); R[f"pope_{pname}_auc_ci"] = [float(lo), float(hi)]
        s = np.array(sc); ok = np.isfinite(s)
        R[f"pope_{pname}_acc0"] = float(((s[ok] > 0) == np.array(ys)[ok]).mean()); R[f"pope_{pname}_yes_rate"] = float((s[ok] > 0).mean())
        R[f"pope_{pname}_nan"] = int((~ok).sum())
        if sn: R["pope_cap_null_auc"] = float(auc_ci(ys, sn)[0])
    BAD_IMGS[label] = bad
    return R

todo = [(k, v) for k, v in CKPTS.items() if k not in RES]
t_all = time.time()
for n, (label, sd) in enumerate(todo):
    t0 = time.time(); RES[label] = eval_ckpt(label, sd)
    RES[label]["seconds"] = time.time() - t0
    json.dump(RES, open(RES_PATH, "w"), indent=1)
    R = RES[label]
    print(f"[{label}] nonfinite {R['nonfinite']}/{R['scan_n']} | heldout {R['heldout_loss']:.3f} train {R['train_loss']:.3f} | "
          f"shuf-real {R['shuf_minus_real_mean']:+.3f} (t={R['shuf_minus_real_t']:.1f}) | POPE AUC cap {R['pope_cap_auc']:.3f} / vic {R['pope_vic_auc']:.3f} (null {R['pope_cap_null_auc']:.3f}) | {R['seconds']/60:.1f} min")
    if n == 0: print(f"   ETA for remaining {len(todo)-1} checkpoints: ~{(len(todo)-1)*R['seconds']/60:.0f} min")
print("done", f"{(time.time()-t_all)/60:.1f} min")

[s6600] nonfinite 0/60 | heldout 3.865 train 3.875 | shuf-real +0.407 (t=5.7) | POPE AUC cap 0.522 / vic 0.518 (null 0.565) | 3.2 min
   ETA for remaining 1 checkpoints: ~3 min
[s6800] nonfinite 0/60 | heldout 3.856 train 3.882 | shuf-real +0.403 (t=5.5) | POPE AUC cap 0.515 / vic 0.490 (null 0.593) | 3.4 min
done 6.5 min


checking by reproducing training set to see if training loss reduces

In [10]:
@torch.no_grad()
def train_pipeline_loss(P, images, captions):
    projected = P(tower(images))
    tk = tokenizer(list(captions), return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    ids, am = tk.input_ids, tk.attention_mask
    emb = torch.cat([projected, llm.get_input_embeddings()(ids)], 1).to(next(llm.parameters()).dtype)
    full_am = torch.cat([torch.ones(am.shape[0], projected.shape[1], device=device, dtype=am.dtype), am], 1)
    lab = torch.cat([torch.full((ids.shape[0], projected.shape[1]), -100, device=device, dtype=ids.dtype), ids.clone()], 1)
    return float(llm(inputs_embeds=emb, attention_mask=full_am, labels=lab).loss)

val_names = (manifest["tuned_lens_val"] + manifest["tuned_lens_test"])[:40]
imgs = {n: Image.open(f"{IMG_DIR}/{n}").convert("RGB") for n in val_names}
print("tokens:", tokenizer.convert_ids_to_tokens(tokenizer(first_cap[val_names[0]]).input_ids[:6]), "| padding_side:", tokenizer.padding_side)

f0 = pope_imgs[0]; live = tower([Image.open(f"{IMG_DIR}/{f0}").convert("RGB")])[0].float().cpu()
print("cached vs live encoder features, max abs diff:", float((FEATS[f0] - live).abs().max()))

for label in CKPTS:
    P = make_projector(CKPTS[label])
    A = [train_pipeline_loss(P, [imgs[a], imgs[b]], [first_cap[a], first_cap[b]]) for a, b in zip(val_names[::2], val_names[1::2])]
    with torch.no_grad():
        B = [fwd(P(tower([imgs[n]])), first_cap[n], mask_bos=False)[0] for n in val_names]
    print(f"{label}: training-pipeline loss {np.nanmean(A):.3f} (nan batches {int(np.isnan(A).sum())}) | my fwd, batch 1, live encoder {np.nanmean(B):.3f}")

tokens: ['<s>', '▁A', '▁yellow', '▁clock', '▁that', '▁is'] | padding_side: right
cached vs live encoder features, max abs diff: 0.0052449703216552734
s6600: training-pipeline loss 12.903 (nan batches 5) | my fwd, batch 1, live encoder 13.580
s6800: training-pipeline loss 11.892 (nan batches 6) | my fwd, batch 1, live encoder 12.362


In [14]:
f = pope_imgs[0]
def stat(h):
    h = h.float(); fin = torch.isfinite(h)
    return float(h[fin].abs().max()) if fin.any() else float("nan"), int((~fin).sum())

for label in CKPTS:
    P = make_projector(CKPTS[label]); e = project(P, f)
    print(f"\n[{label}] projector out max|x| {float(e.abs().max()):.1f} | finite after fp16 cast: {bool(torch.isfinite(e.to(LLM_DTYPE)).all())}")
    for pname, txt in [("caption", first_cap[f]), ("cap", PROMPTS["cap"]("dog")), ("vic", PROMPTS["vic"]("dog"))]:
        ids = tokenizer(txt, return_tensors="pt").input_ids.to(device)
        emb = torch.cat([e, _embed(ids)], 1).to(LLM_DTYPE)
        with torch.no_grad():
            out = llm(inputs_embeds=emb, attention_mask=torch.ones(emb.shape[:2], dtype=torch.long, device=device), output_hidden_states=True)
        hs = out.hidden_states
        bad = next((i for i, h in enumerate(hs) if not torch.isfinite(h).all()), None)
        row = {i: stat(hs[i]) for i in (0, 4, 8, 16, 24, 32)}
        print(f"  {pname:8s} seq {emb.shape[1]:4d} | logits finite {bool(torch.isfinite(out.logits).all())} | last-pos finite {bool(torch.isfinite(out.logits[0,-1]).all())} | first bad layer {bad}")
        print("           (max finite |h|, n non-finite) by layer:", {i: (round(a), n) for i, (a, n) in row.items()})

s6600 {'scan_n': 60, 'nonfinite': 3, 'proj_max_abs_max': 320.294, 'norm_ratio_vs_text': 1338.587, 'heldout_loss': 14.267, 'shuf_minus_real_mean': 0.608, 'train_loss': 14.618, 'pope_cap_auc': 0.515, 'pope_cap_nan': 0, 'pope_vic_auc': 1.0, 'pope_vic_nan': 38}
s6800 {'scan_n': 60, 'nonfinite': 7, 'proj_max_abs_max': 336.612, 'norm_ratio_vs_text': 1380.408, 'heldout_loss': 12.935, 'shuf_minus_real_mean': 0.667, 'train_loss': 13.324, 'pope_cap_auc': 0.548, 'pope_cap_nan': 0, 'pope_vic_auc': nan, 'pope_vic_nan': 40}
\n[s6600] projector out max|x| 273.6 | finite after fp16 cast: True
  caption  seq  802 | all logits finite True | last-pos finite True | first bad layer None | max|h| @ layers 0/8/16/24/32: [274, 1950, 1958, 1968, 83]
  cap      seq  797 | all logits finite True | last-pos finite True | first bad layer None | max|h| @ layers 0/8/16/24/32: [274, 1950, 1958, 1968, 83]


ValueError: cannot convert float NaN to integer

In [16]:
def probe(img_emb, txt, name):
    ids = tokenizer(txt, return_tensors="pt").input_ids.to(device)
    emb = torch.cat([img_emb, _embed(ids)], 1) if img_emb is not None else _embed(ids)
    emb = emb.to(LLM_DTYPE); n_img = 0 if img_emb is None else img_emb.shape[1]
    with torch.no_grad():
        out = llm(inputs_embeds=emb, attention_mask=torch.ones(emb.shape[:2], dtype=torch.long, device=device), output_hidden_states=True)
    hs = out.hidden_states
    first_bad = next((i for i, h in enumerate(hs) if not torch.isfinite(h).all()), None)
    badpos = (~torch.isfinite(hs[-1]).all(-1))[0].nonzero().flatten().tolist()
    rel = [p - n_img for p in badpos]
    print(f"{name:34s} tokens {emb.shape[1]:4d} | last-pos finite {bool(torch.isfinite(out.logits[0,-1]).all())} | first bad layer {first_bad} | bad positions (rel. to text start): {rel[:8]}{'...' if len(rel)>8 else ''} (n={len(rel)})")

label = list(CKPTS)[0]; P = make_projector(CKPTS[label])
e0, e1 = project(P, pope_imgs[0]), project(P, pope_imgs[1])
q = "Is there a dog in the image? Please answer yes or no."
probe(e0, PROMPTS["vic"]("dog"),               "image0 + full vic prompt")
probe(e1, PROMPTS["vic"]("dog"),               "image1 + full vic prompt")
probe(None, PROMPTS["vic"]("dog"),             "NO image + full vic prompt")
probe(e0, SYS,                                 "image0 + system prompt only")
probe(e0, f"USER: {q} ASSISTANT:",             "image0 + short USER/ASSISTANT")
probe(e0, f"Question: {q} Answer:",            "image0 + cap-style, same text")

image0 + full vic prompt           tokens  837 | last-pos finite False | first bad layer 31 | bad positions (rel. to text start): [-16, -15, -14, -13, -12, -11, -10, -9]... (n=69)
image1 + full vic prompt           tokens  837 | last-pos finite False | first bad layer 31 | bad positions (rel. to text start): [-16, -15, -14, -13, -12, -11, -10, -9]... (n=69)
NO image + full vic prompt         tokens   53 | last-pos finite True | first bad layer None | bad positions (rel. to text start): [] (n=0)
image0 + system prompt only        tokens  816 | last-pos finite False | first bad layer 31 | bad positions (rel. to text start): [-16, -15, -14, -13, -12, -11, -10, -9]... (n=48)
image0 + short USER/ASSISTANT      tokens  807 | last-pos finite True | first bad layer None | bad positions (rel. to text start): [] (n=0)
image0 + cap-style, same text      tokens  803 | last-pos finite True | first bad layer None | bad positions (rel. to text start): [] (n=0)


In [17]:
print("attn impl:", getattr(llm.config, "_attn_implementation", None))
label = list(CKPTS)[0]; P = make_projector(CKPTS[label]); e0 = project(P, pope_imgs[0])
def sweep(tag):
    res = []
    for n in range(8, 70, 4):                       # text tokens = 1 BOS + n
        ids = tokenizer("a " * n, return_tensors="pt").input_ids.to(device)
        emb = torch.cat([e0, _embed(ids)], 1).to(LLM_DTYPE)
        with torch.no_grad():
            lg = llm(inputs_embeds=emb, attention_mask=torch.ones(emb.shape[:2], dtype=torch.long, device=device)).logits
        res.append((emb.shape[1], bool(torch.isfinite(lg).all())))
    print(tag, res)
sweep("default attention :")
orig = llm.config._attn_implementation
for impl in ["eager"]:
    try:
        llm.config._attn_implementation = impl; sweep(f"{impl:17s}:")
    except Exception as ex: print(impl, "failed:", ex)
llm.config._attn_implementation = orig

attn impl: sdpa
default attention : [(794, True), (798, True), (802, False), (806, True), (810, True), (814, True), (818, True), (822, True), (826, True), (830, True), (834, True), (838, True), (842, True), (846, True), (850, True), (854, True)]
eager            : [(794, True), (798, True), (802, False), (806, True), (810, True), (814, True), (818, True), (822, True), (826, True), (830, True), (834, True), (838, True), (842, True), (846, True), (850, True), (854, True)]


In [18]:
lens = {f: len(tokenizer(first_cap[f]).input_ids) for f in pope_imgs}
print("caption token length (incl BOS): min/median/max =", min(lens.values()), int(np.median(list(lens.values()))), max(lens.values()), "| >=25 tokens:", sum(v >= 25 for v in lens.values()), "of", len(lens))
for k, R in RES.items():
    print(k, "non-finite images:", len(R["nonfinite_imgs"]), "-> their caption lengths:", sorted(lens[f] for f in R["nonfinite_imgs"]))

caption token length (incl BOS): min/median/max = 9 13 24 | >=25 tokens: 0 of 60
s6600 non-finite images: 3 -> their caption lengths: [12, 13, 14]
s6800 non-finite images: 7 -> their caption lengths: [11, 12, 12, 13, 14, 14, 14]


In [19]:
def diag(P, f, pad_extra=0):
    e = project(P, f); ids = tokenizer(first_cap[f], return_tensors="pt").input_ids.to(device)
    emb = torch.cat([e, _embed(ids)], 1).to(LLM_DTYPE); am = torch.ones(emb.shape[:2], dtype=torch.long, device=device)
    if pad_extra:
        pad = _embed(torch.full((1, pad_extra), tokenizer.pad_token_id, device=device)).to(LLM_DTYPE)
        emb = torch.cat([emb, pad], 1); am = torch.cat([am, torch.zeros(1, pad_extra, dtype=torch.long, device=device)], 1)
    with torch.no_grad(): out = llm(inputs_embeds=emb, attention_mask=am, output_hidden_states=True)
    n_img, n_txt = e.shape[1], ids.shape[1]
    rows = (~torch.isfinite(out.logits[0].float()).all(-1)).nonzero().flatten().tolist()
    fb = next((i for i, h in enumerate(out.hidden_states) if not torch.isfinite(h).all()), None)
    lab = torch.cat([torch.full((1, n_img), -100, device=device), ids], 1); lab[:, n_img] = -100
    loss = float(torch.nn.functional.cross_entropy(out.logits[0, :n_img+n_txt-1].float(), lab[0, 1:], ignore_index=-100))
    return dict(seq=emb.shape[1], loss_finite=math.isfinite(loss), bad_rows=len(rows),
                in_image=sum(p < n_img for p in rows), in_text=sum(n_img <= p < n_img + n_txt for p in rows), first_bad_layer=fb)

for label, R in RES.items():
    P = make_projector(CKPTS[label])
    print(f"\n== {label}: {len(R['nonfinite_imgs'])} flagged images")
    for f in R["nonfinite_imgs"]:
        print(f[-16:-4], {pe: diag(P, f, pe) for pe in (0, 1, 2)})


== s6600: 3 flagged images
000000016451 {0: {'seq': 797, 'loss_finite': False, 'bad_rows': 29, 'in_image': 16, 'in_text': 13, 'first_bad_layer': 31}, 1: {'seq': 798, 'loss_finite': False, 'bad_rows': 798, 'in_image': 784, 'in_text': 13, 'first_bad_layer': 31}, 2: {'seq': 799, 'loss_finite': False, 'bad_rows': 799, 'in_image': 784, 'in_text': 13, 'first_bad_layer': 31}}
000000023084 {0: {'seq': 798, 'loss_finite': False, 'bad_rows': 30, 'in_image': 16, 'in_text': 14, 'first_bad_layer': 31}, 1: {'seq': 799, 'loss_finite': False, 'bad_rows': 799, 'in_image': 784, 'in_text': 14, 'first_bad_layer': 31}, 2: {'seq': 800, 'loss_finite': False, 'bad_rows': 800, 'in_image': 784, 'in_text': 14, 'first_bad_layer': 31}}
000000052689 {0: {'seq': 796, 'loss_finite': False, 'bad_rows': 28, 'in_image': 16, 'in_text': 12, 'first_bad_layer': 31}, 1: {'seq': 797, 'loss_finite': False, 'bad_rows': 797, 'in_image': 784, 'in_text': 12, 'first_bad_layer': 31}, 2: {'seq': 798, 'loss_finite': False, 'bad_rows'

Test A: is it just fp16 overflow? Rerun the 10 flagged images with Vicuna computing in fp32. If they all come out finite, the weights are fine and the problem is only the 4-bit fp16 compute, which would be a much better situation.

In [20]:
import gc
llm32 = AutoModelForCausalLM.from_pretrained(
    LLM_PATH, torch_dtype=torch.float32, device_map={"": 0}, low_cpu_mem_usage=True,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float32)).eval()
for p in llm32.parameters(): p.requires_grad_(False)

@torch.no_grad()
def fwd32(P, f):
    e = project(P, f); ids = tokenizer(first_cap[f], return_tensors="pt").input_ids.to(device)
    emb = torch.cat([e, llm32.get_input_embeddings()(ids)], 1).float()
    lab = torch.cat([torch.full((1, e.shape[1]), -100, device=device), ids], 1); lab[:, e.shape[1]] = -100
    out = llm32(inputs_embeds=emb, attention_mask=torch.ones(emb.shape[:2], dtype=torch.long, device=device), labels=lab)
    return float(out.loss), bool(torch.isfinite(out.logits).all())

for label, R in RES.items():
    P = make_projector(CKPTS[label])
    print(label, [(f[-10:-4], *[round(x, 3) if isinstance(x, float) else x for x in fwd32(P, f)]) for f in R["nonfinite_imgs"]])

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

s6600 [('016451', 11.942, True), ('023084', 11.532, True), ('052689', 11.497, True)]
s6800 [('006033', 11.781, True), ('011241', 11.347, True), ('016451', 10.429, True), ('023084', 10.148, True), ('032610', 12.865, True), ('052689', 10.539, True), ('068418', 10.558, True)]


In [1]:
flag = sorted({f for R in RES.values() for f in R["nonfinite_imgs"]})
ok = [f for f in pope_imgs if f not in flag][:15]
label = "s6600"; P = make_projector(CKPTS[label])

# 1) control: does fp32 give normal loss on images that were never flagged?
l_ok  = [fwd32(P, f)[0] for f in ok]
l_bad = [fwd32(P, f)[0] for f in flag if f in set(RES[label]["nonfinite_imgs"])]
print(f"fp32 loss, {len(ok)} normal images: mean {np.mean(l_ok):.3f} (range {min(l_ok):.2f}-{max(l_ok):.2f}) | flagged: mean {np.mean(l_bad):.3f}")

# 2) token-norm stats, encoder output and projector output
@torch.no_grad()
def tokstats(f):
    fn = FEATS[f].norm(dim=-1); en = project(P, f)[0].norm(dim=-1)
    return dict(enc_max=round(float(fn.max()), 1), enc_med=round(float(fn.median()), 1), proj_max=round(float(en.max()), 1),
                proj_med=round(float(en.median()), 1), max_at=int(en.argmax()))
print("\nNORMAL ", [tokstats(f) for f in ok[:6]])
print("\nFLAGGED", [tokstats(f) for f in flag[:6]])

NameError: name 'RES' is not defined

## 6. Summary table and checkpoint choice

In [ ]:
import pandas as pd
df = pd.DataFrame(RES).T
df["step"] = [step_of(k) for k in df.index]
df = df.sort_values("step")
cols = ["step", "nonfinite", "proj_max_abs_max", "norm_ratio_vs_text", "heldout_loss", "train_loss", "shuf_minus_real_mean",
        "shuf_minus_real_t", "noimg_minus_real_mean", "pope_cap_auc", "pope_cap_null_auc", "pope_vic_auc", "pope_cap_yes_rate", "pope_vic_yes_rate"]
pd.set_option("display.width", 250, "display.max_columns", 50)
print(df[cols].round(3).to_string())
df[cols].round(4).to_csv(f"{OUT_DIR}/summary.csv")

elig = df[(df["nonfinite"] == 0) & (df["train_loss"].notna())]
print("\nEligible (0 non-finite over", int(df["scan_n"].iloc[0]), "POPE images + train images):", list(elig.index))
if len(elig):
    pick = elig["heldout_loss"].idxmin()
    safest = elig["proj_max_abs_max"].idxmin()
    print(f"best held-out loss among eligible : {pick}  (heldout {elig.loc[pick,'heldout_loss']:.3f})")
    print(f"smallest projector output (safest): {safest}  (max|x| {elig.loc[safest,'proj_max_abs_max']:.1f})")
    print(f"latest eligible                   : {elig.index[-1]}")
else:
    print("NO checkpoint is NaN-free on the POPE-manifest images. See section 7: fp16 check / lower-LR retrain needed.")
    pick = df["nonfinite"].idxmin(); print("fewest non-finite:", pick, int(df.loc[pick, "nonfinite"]))

import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
ax[0].plot(df["step"], df["heldout_loss"], "o-", label="held-out (POPE imgs)"); ax[0].plot(df["step"], df["train_loss"], "s--", label="train split")
ax[0].axhline(np.mean(list(noimg.values())), color="gray", ls=":", label="no-image"); ax[0].set_title("caption loss (BOS masked, batch 1)"); ax[0].legend(); ax[0].set_xlabel("step")
ax[1].plot(df["step"], df["nonfinite"], "o-", color="C3"); ax[1].set_title("non-finite images / %d" % int(df["scan_n"].iloc[0])); ax[1].set_xlabel("step")
ax[2].plot(df["step"], df["pope_cap_auc"], "o-", label="real"); ax[2].plot(df["step"], df["pope_cap_null_auc"], "x--", label="shuffled-image null"); ax[2].plot(df["step"], df["pope_vic_auc"], "s-", label="vicuna prompt")
ax[2].axhline(0.5, color="gray", ls=":"); ax[2].set_title("mini-POPE AUC"); ax[2].legend(); ax[2].set_xlabel("step")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/summary.png", dpi=130); plt.show()

## 7. Verdict on usability (read this before using any checkpoint)

In [ ]:
def verdict(label):
    R = RES[label]; msg = []
    t = R["shuf_minus_real_t"]; g = R["shuf_minus_real_mean"]
    msg.append(f"GROUNDING (loss): shuffled-image loss is {g:+.3f} vs real (paired t={t:.1f}) -> " + ("image content matters to the LM" if t > 3 and g > 0.05 else "NOT clearly grounded: shuffled image is about as good as the right one"))
    msg.append(f"IMAGE vs NONE   : no-image loss is {R['noimg_minus_real_mean']:+.3f} above real -> " + ("projector helps" if R["noimg_minus_real_mean"] > 0.05 else "projector does NOT beat having no image"))
    gap = R["train_loss"] - R["heldout_loss"]
    msg.append(f"GENERALISATION  : train {R['train_loss']:.3f} vs held-out {R['heldout_loss']:.3f} (gap {R['heldout_loss']-R['train_loss']:+.3f}) -> " + ("overfit" if R["heldout_loss"] - R["train_loss"] > 0.3 else "no large overfit"))
    lo, hi = R["pope_cap_auc_ci"]
    msg.append(f"MINI-POPE (cap) : AUC {R['pope_cap_auc']:.3f} [{lo:.2f},{hi:.2f}] vs shuffled-null {R['pope_cap_null_auc']:.3f}; yes-rate {R['pope_cap_yes_rate']:.2f} -> " + ("above chance" if lo > 0.55 else "indistinguishable from chance (expected for caption-only training; needs instruction tuning)"))
    msg.append(f"STABILITY       : {R['nonfinite']}/{R['scan_n']} non-finite; projector |x|max {R['proj_max_abs_max']:.0f}; token-norm / text-embed-norm = {R['norm_ratio_vs_text']:.1f}x")
    return "\n   ".join(msg)
for lab in (list(elig.index[-3:]) if len(elig) else [df.index[-1]]):
    print(f"== {lab}\n   " + verdict(lab) + "\n")

## 8. Caption quality (Gate 4) — top eligible checkpoints only

In [ ]:
GEN = {}
cands = list(elig.sort_values("heldout_loss").index[:GEN_TOPK]) if len(elig) else []
G_NAMES = pope_imgs[:GEN_N]
for label in cands:
    P = make_projector(CKPTS[label]); rows_ = []
    r = random.Random(SEED + 11); perm = G_NAMES[:]; r.shuffle(perm)
    for f, g in zip(G_NAMES, perm):
        c_real = gen_caption(project(P, f)); c_shuf = gen_caption(project(P, g if g != f else G_NAMES[(G_NAMES.index(f)+1) % len(G_NAMES)]))
        rows_.append((f, c_real, c_shuf))
    def pr(caps, key):
        tp = fp = fn = 0
        for f, *cc in caps:
            m = mentioned_cats(cc[key]); gt = gt_cats[f]; tp += len(m & gt); fp += len(m - gt); fn += len(gt - m)
        return tp / max(tp + fp, 1), tp / max(tp + fn, 1), tp + fp
    p1, r1, n1 = pr(rows_, 0); p2, r2, n2 = pr(rows_, 1)
    GEN[label] = dict(real_prec=p1, real_rec=r1, shuf_prec=p2, shuf_rec=r2, n_real_mentions=n1, n_shuf_mentions=n2, examples=rows_[:8])
    print(f"[{label}] object precision real {p1:.2f} vs shuffled-image {p2:.2f} | recall real {r1:.2f} vs shuffled {r2:.2f} | mentions {n1}/{n2}")
json.dump({k: {kk: vv for kk, vv in v.items()} for k, v in GEN.items()}, open(f"{OUT_DIR}/gen.json", "w"), indent=1)
for label in cands[:1]:
    print(f"\nSample captions for {label}:")
    for f, a, b in GEN[label]["examples"]:
        print(f"  {f[-16:-4]} | GT: {sorted(gt_cats[f])[:5]}\n      real : {a}\n      shuf : {b}")

## 9. Side checks: pad-token loss bug size, optional fp16-Vicuna cross-check

In [ ]:
# (a) the training script set labels = input_ids.clone() with pad token = <unk> (or eos) and batch 2 -> pad positions were trained on
best = (pick if len(elig) else df.index[-1]); P = make_projector(CKPTS[best]); a_tr, a_clean = [], []
with torch.no_grad():
    for i in range(0, 40, 2):
        names_ = pope_imgs[i:i+2]
        if len(names_) < 2: break
        feats = torch.cat([P(FEATS[n].unsqueeze(0).to(device)) for n in names_], 0)
        tk = tokenizer([first_cap[n] for n in names_], return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
        te = _embed(tk.input_ids); emb = torch.cat([feats, te], 1).to(LLM_DTYPE)
        am = torch.cat([torch.ones(2, feats.shape[1], device=device, dtype=tk.attention_mask.dtype), tk.attention_mask], 1)
        lab_tr = torch.cat([torch.full((2, feats.shape[1]), -100, device=device), tk.input_ids], 1)
        lab_cl = lab_tr.clone(); lab_cl[:, feats.shape[1]:][tk.attention_mask == 0] = -100
        a_tr.append(float(llm(inputs_embeds=emb, attention_mask=am, labels=lab_tr).loss)); a_clean.append(float(llm(inputs_embeds=emb, attention_mask=am, labels=lab_cl).loss))
print(f"[{best}] batch-2 loss as trained (pads in loss): {np.nanmean(a_tr):.3f} | pads masked: {np.nanmean(a_clean):.3f} | padding_side={tokenizer.padding_side}")
print("-> if these differ, the logged train/val numbers (~3.8-4.0) were inflated/biased by pad-token terms; the BOS-masked batch-1 numbers above are the clean ones.")

# (b) optional: does an fp16 (unquantised) Vicuna remove the non-finite images?
if RUN_FP16_CHECK and torch.cuda.device_count() > 1:
    llm16 = AutoModelForCausalLM.from_pretrained(LLM_PATH, torch_dtype=torch.float16, device_map={"": 1}).eval()
    for label, imgs_bad in BAD_IMGS.items():
        if not imgs_bad: continue
        P = make_projector(CKPTS[label]); ok16 = 0
        for f in imgs_bad[:20]:
            e = project(P, f).to("cuda:1", torch.float16); ids = tokenizer(first_cap[f], return_tensors="pt").input_ids.to("cuda:1")
            emb = torch.cat([e, llm16.get_input_embeddings()(ids)], 1)
            ok16 += int(torch.isfinite(llm16(inputs_embeds=emb).logits).all())
        print(f"[{label}] images non-finite under 4-bit: {len(imgs_bad)}; of first {min(20,len(imgs_bad))}, finite under fp16 Vicuna: {ok16}")
else:
    print("fp16 cross-check skipped (set RUN_FP16_CHECK=True with GPU T4 x2 and non-finite images present)")

## How to read the result

* **Which step:** use the *latest checkpoint that is NaN-free on all 500 POPE images* **and** whose held-out loss is near the minimum. A lower held-out loss at a step that fails the stability scan is not usable — the NaNs mean cache extraction on those inputs would break.
* **Is it usable at all:** the key evidence is Gate 3/4 — the shuffled-image loss must be clearly worse than the real-image loss (paired t > 3), and generated-caption object precision must beat the shuffled-image captions. If not, the projector has learned a language prior, not vision, and the CLIP-vs-JEPA comparison would be comparing against noise.
* **POPE-style Yes/No:** AUC near 0.5 is *expected* here (caption-only training, no instruction tuning); it only tells you whether stage-2 tuning is required, not that the projector is broken.
* Upload `summary.csv`, `summary.png`, `results.json`, `gen.json` from `/kaggle/working/projector_analysis/` back to the chat.